In [ ]:
import os
import re
import polars as pl
from sqlalchemy import create_engine
import time


RAW = "data/raw"
CUTOFF = 15            # a flight is considered "delayed" if it arrives > 15 min late (DOT standard)
FAP_TOL = 30           # a shipment/cargo is "on plan" if it slips <= 30 min

SERVER   = os.getenv("SQL_SERVER", "MS\SQLEXPRESS")
DATABASE = os.getenv("SQL_DATABASE", "CargoOps")
DRIVER   = "ODBC Driver 17 for SQL Server"
CONN = (f"mssql+pyodbc://@{SERVER}/{DATABASE}"
        f"?driver={DRIVER.replace(' ', '+')}&trusted_connection=yes")


# creating below 4 containers to store/ append result data
tables = {}
all_carriers = []
all_routes = []
all_dates = []

# transforming flights data

def load_flights():
    path = f"{RAW}/flights.csv"
    if not os.path.exists(path):
        print("flights.csv not found")
        return

    df = pl.read_csv(path, infer_schema_length=10000)

    rename_map = {
        "YEAR": "year", "MONTH": "month", "DAY": "day",
        "AIRLINE": "carrier",
        "ORIGIN_AIRPORT": "origin", "DESTINATION_AIRPORT": "dest",
        "ARRIVAL_DELAY": "arr_delay", "DEPARTURE_DELAY": "dep_delay",
        "AIRLINE_DELAY": "carrier_delay", "WEATHER_DELAY": "weather_delay",
        "AIR_SYSTEM_DELAY": "nas_delay", "SECURITY_DELAY": "security_delay",
        "LATE_AIRCRAFT_DELAY": "late_aircraft_delay",
    }
    df = df.select(list(rename_map.keys())).rename(rename_map)
    df = df.drop_nulls(["carrier", "origin", "dest", "arr_delay"])

    df = df.with_columns(pl.date("year", "month", "day").alias("flight_date"))
    df = df.with_columns((pl.col("origin") + "->" + pl.col("dest")).alias("route"))
    df = df.with_columns((pl.col("arr_delay") > CUTOFF).cast(pl.Int8).alias("delayed"))
    df = df.with_columns((1 - pl.col("delayed")).alias("on_time"))

    cause_cols = ["carrier_delay", "weather_delay", "nas_delay",
                  "security_delay", "late_aircraft_delay"]
    df = df.with_columns(pl.col(cause_cols).fill_null(0))

    biggest = pl.max_horizontal(cause_cols)
    cause = pl.when(pl.col("delayed") == 0).then(pl.lit("On-Time"))
    nice_names = ["Carrier", "Weather", "NAS", "Security", "Late Aircraft"]
    for col, label in zip(cause_cols, nice_names):
        cause = cause.when(pl.col(col) == biggest).then(pl.lit(label))
    cause = cause.otherwise(pl.lit("Other"))
    df = df.with_columns(cause.alias("delay_cause"))

    all_carriers.append(df.select("carrier").unique())
    all_routes.append(df.select("route", "origin", "dest").unique())
    all_dates.append(df.select("flight_date").unique())

    tables["fact_flights"] = df.select(
        "flight_date", "carrier", "route", "arr_delay", "dep_delay",
        "delayed", "on_time", "delay_cause")
    print(f"flights data ready: {tables['fact_flights'].height:,} flights")

    

#renaming cargo milestone KPIs

def milestone_name(column):
    """ renaming 'i1_rcs_p' -> 'rcs' ; 'o_dlv_e' -> 'dlv' ; returns None if no match """
    match = re.match(r"^(?:i\d+|o)_([a-z]+)", column)
    return match.group(1) if match else None

# transforming cargo iq data

def load_cargo_iq():
    path = f"{RAW}/c2k.csv"
    if not os.path.exists(path):
        print("c2k.csv not found")
        return

    df = pl.read_csv(path, infer_schema_length=10000)

    # pairing each planned (_p) column with its effective/ corresponding (_e) twin
    planned_cols, effective_cols = [], []
    for col in df.columns:
        if col.endswith("_p") and milestone_name(col):
            twin = col[:-2] + "_e"
            if twin in df.columns:
                planned_cols.append(col)
                effective_cols.append(twin)

    # making every minute column numeric; filling NULL values with 0
    for col in planned_cols + effective_cols:
        df = df.with_columns(pl.col(col).cast(pl.Float64, strict=False).fill_null(0))

    df = df.with_columns(pl.sum_horizontal(planned_cols).alias("planned_min"))
    df = df.with_columns(pl.sum_horizontal(effective_cols).alias("effective_min"))
    df = df.with_columns((pl.col("effective_min") - pl.col("planned_min")).alias("slippage_min"))

    df = df.with_columns((pl.col("slippage_min") <= FAP_TOL).cast(pl.Int8).alias("fap"))
    df = df.with_columns((pl.col("slippage_min") <= 0).cast(pl.Int8).alias("dap"))

    # Calculating slippage per-milestone (where time is lost)
    milestones = sorted(set(milestone_name(c) for c in planned_cols))
    for m in milestones:
        m_planned = [c for c in planned_cols if milestone_name(c) == m]
        m_effective = [c for c in effective_cols if milestone_name(c) == m]
        var = pl.sum_horizontal(m_effective) - pl.sum_horizontal(m_planned)
        df = df.with_columns(var.alias(f"var_{m}"))

    # Calculating Dwell Time which is ground time between arrival (RCF) and delivery (DLV)
    rcf_actual = [c for c in effective_cols if milestone_name(c) == "rcf"]
    dlv_actual = [c for c in effective_cols if milestone_name(c) == "dlv"]
    has_dwell = bool(rcf_actual and dlv_actual)
    if has_dwell:
        df = df.with_columns(
            (pl.sum_horizontal(dlv_actual) - pl.sum_horizontal(rcf_actual)).alias("dwell_min"))
        
    # Creating imaginery Hub names as the real file masks airport codes as numbers.
    
    HUB_NAMES = ["Hub-DOH", "Hub-FRA", "Hub-HKG", "Hub-LHR",
                 "Hub-DXB", "Hub-SIN", "Hub-AMS", "Hub-ICN"]
    place_cols = [c for c in df.columns if c.endswith("_place")]
    has_hub = False
    if place_cols:
        hub_src = next((c for c in place_cols if c.startswith("o_")), place_cols[0])
        df = df.with_columns(
            pl.col(hub_src).cast(pl.Int64, strict=False).fill_null(0).alias("_pid"))
        df = df.with_columns(
            pl.col("_pid").map_elements(
                lambda x: HUB_NAMES[int(x) % len(HUB_NAMES)],
                return_dtype=pl.String).alias("hub"))
        has_hub = True
            
    keep = ["planned_min", "effective_min", "slippage_min", "fap", "dap"]
    keep += [f"var_{m}" for m in milestones]
    if has_dwell:
        keep.append("dwell_min")
    if has_hub:                          # using the new imaginary hub
        keep = ["hub"] + keep
    elif "o_hub" in df.columns:          # fallback for the synthetic test file
        df = df.rename({"o_hub": "hub"})
        keep = ["hub"] + keep    

    tables["fact_cargo_iq"] = df.select(keep)
    print(f"cargo_iq ready: {tables['fact_cargo_iq'].height:,} shipments")


# transforming cargo tonnage data taken from t100 BTS

def load_tonnage():
    path = f"{RAW}/t100_cargo.csv"
    if not os.path.exists(path):
        print("t100_cargo.csv not found")
        return

    df = pl.read_csv(path, infer_schema_length=10000)

    rename_map = {
        "UNIQUE_CARRIER": "carrier", "ORIGIN": "origin", "DEST": "dest",
        "FREIGHT": "freight_lb", "MAIL": "mail_lb",
        "PAYLOAD": "capacity_lb",      # available capacity (for load factor)
        "DISTANCE": "distance_mi",     # route length (for yield / FTK)
        "YEAR": "year", "MONTH": "month",
    }
    df = df.select(list(rename_map.keys())).rename(rename_map)
    df = df.drop_nulls(["carrier", "origin", "dest"])

    df = df.with_columns((pl.col("origin") + "->" + pl.col("dest")).alias("route"))
    df = df.with_columns((pl.col("freight_lb").fill_null(0) / 2204.62).alias("freight_tonnes"))
    df = df.with_columns((pl.col("mail_lb").fill_null(0) / 2204.62).alias("mail_tonnes"))
    df = df.with_columns((pl.col("capacity_lb").fill_null(0) / 2204.62).alias("capacity_tonnes"))
    df = df.with_columns(pl.date("year", "month", 1).alias("flight_date"))

    # Calculating Freight Tonne Kilometer (FTK) = tonnes carried x distance flown (miles is converted to km)
    df = df.with_columns(
        (pl.col("freight_tonnes") * pl.col("distance_mi") * 1.60934).alias("ftk"))
    
    # Calculating AFTK
    df = df.with_columns(
    (pl.col("capacity_tonnes") * pl.col("distance_mi") * 1.60934).alias("aftk"))

    # Calculating Cargo Yield: simulated rate per tonne-km based on actual cargo yield figure in IATA data
    df = df.with_columns(
        (0.45 - 0.00005 * pl.col("distance_mi")).clip(0.12, 0.45).alias("yield_per_ftk"))

    # estimated revenue = ftk  x the rate
    df = df.with_columns((pl.col("ftk") * pl.col("yield_per_ftk")).alias("est_revenue"))

    all_carriers.append(df.select("carrier").unique())
    all_routes.append(df.select("route", "origin", "dest").unique())
    all_dates.append(df.select("flight_date").unique())

    tables["fact_tonnage"] = df.select(
        "flight_date", "carrier", "route", "freight_tonnes", "mail_tonnes",
        "capacity_tonnes","ftk","aftk", "yield_per_ftk", "est_revenue")
    print(f"cargo_tonnage: {tables['fact_tonnage'].height:,} tonnage rows")


#creating dim tables
    
def build_dimensions():
    if all_carriers:
        carriers = pl.concat(all_carriers).unique().sort("carrier")
        tables["dim_carrier"] = carriers.with_row_index("carrier_id", offset=1)
    if all_routes:
        routes = pl.concat(all_routes).unique().sort("route")
        tables["dim_route"] = routes.with_row_index("route_id", offset=1)
    if all_dates:
        dates = pl.concat(all_dates).unique().sort("flight_date")
        tables["dim_date"] = dates.with_columns(
            pl.col("flight_date").dt.year().alias("year"),
            pl.col("flight_date").dt.month().alias("month"),
            pl.col("flight_date").dt.strftime("%b").alias("month_name"),
            pl.col("flight_date").dt.quarter().alias("quarter"))
    print("dimensions built")


#write to SQL Server - Gold layer

def write_to_sql():
    if not tables:
        print("nothing to load")
        return
    engine = create_engine(CONN, fast_executemany=True)
    order = ["dim_carrier", "dim_route", "dim_date",
             "fact_flights", "fact_cargo_iq", "fact_tonnage"]
    
    with engine.begin() as conn:
        for name in order:
            if name in tables:
                start = time.time()                       # start timer
                tables[name].to_pandas().to_sql(
                    name, conn, if_exists="replace", index=False, chunksize=50000)
                elapsed = time.time() - start             # stop timer
                print(f"loaded {name:<14} {tables[name].height:>8,} rows  in {elapsed:6.1f}s")                
                
# run everything
if __name__ == "__main__":
    steps = [load_flights, load_cargo_iq, load_tonnage, build_dimensions, write_to_sql]
    for step in steps:
        start = time.time()
        step()
    print(f"  {step.__name__:<18} took {time.time() - start:6.1f}s")
    print("all done")

flights data ready: 5,714,008 flights
cargo_iq ready: 3,943 shipments
cargo_tonnage: 418,475 tonnage rows
dimensions built
loaded dim_carrier         126 rows  in    2.4s
loaded dim_route        33,488 rows  in    1.7s
loaded dim_date            365 rows  in    0.7s
loaded fact_flights   5,714,008 rows  in  220.8s
loaded fact_cargo_iq     3,943 rows  in    1.1s
loaded fact_tonnage    418,475 rows  in   14.0s
  write_to_sql       took  241.3s
all done
